# Relative Valuation

Relative valuation estimates what a company may be worth by comparing it with similar companies. Instead of valuing cash flows directly, it asks what investors are paying for comparable earnings, cash flow, sales, or book value.

Abbreviations used in this notebook:

- **P/E**: Price to Earnings, share price divided by earnings per share.
- **EPS**: Earnings Per Share, net income divided by shares outstanding.
- **EV**: Enterprise Value, equity value plus net debt and other claims.
- **EBITDA**: Earnings Before Interest, Taxes, Depreciation, and Amortization.
- **EBIT**: Earnings Before Interest and Taxes.
- **EV/EBITDA**: Enterprise value divided by EBITDA.
- **P/S**: Price to Sales, market capitalization divided by revenue.
- **P/B**: Price to Book, market capitalization divided by book equity.
- **PEG**: Price/Earnings to Growth, P/E divided by expected earnings growth.
- **CHF**: Swiss franc, the currency used in the examples.

## 1. Intuition

DCF valuation asks, "What are the cash flows worth?" Relative valuation asks, "What are similar companies worth in the market?"

A multiple is a shortcut that compares value to a business metric. For example, a company trading at 20x earnings is valued at twenty times its annual net income.

Relative valuation is useful because markets often price similar businesses in similar ways. It is risky because no two companies are perfectly identical. Growth, margins, leverage, risk, geography, and accounting quality all affect the multiple a company deserves.

## 2. Mathematics

Price to earnings:

$$
P/E = \frac{\text{Share Price}}{EPS} = \frac{\text{Market Cap}}{\text{Net Income}}
$$

Enterprise value:

$$
EV = \text{Market Cap} + \text{Net Debt}
$$

Enterprise value to EBITDA:

$$
EV/EBITDA = \frac{EV}{EBITDA}
$$

Price to sales:

$$
P/S = \frac{\text{Market Cap}}{\text{Revenue}}
$$

Price to book:

$$
P/B = \frac{\text{Market Cap}}{\text{Book Equity}}
$$

Implied equity value from an enterprise multiple:

$$
\text{Equity Value} = (\text{Target Metric} \times \text{Peer Multiple}) - \text{Net Debt}
$$

Implied share price:

$$
\text{Implied Share Price} = \frac{\text{Implied Equity Value}}{\text{Shares Outstanding}}
$$

## 3. Implementation

We will build a synthetic peer group for large consumer staples companies. The numbers are simplified and expressed in CHF millions, but the workflow mirrors a real peer comparison.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

peers = pd.DataFrame({
    "company": ["Nestle-like Target", "Peer A", "Peer B", "Peer C", "Peer D", "Peer E"],
    "revenue": [102_000, 86_000, 78_000, 66_000, 54_000, 48_000],
    "ebitda": [22_400, 18_900, 16_200, 13_000, 10_900, 9_100],
    "ebit": [17_500, 14_600, 12_300, 9_700, 8_000, 6_600],
    "net_income": [12_100, 10_000, 8_500, 6_900, 5_500, 4_300],
    "book_equity": [68_600, 55_000, 49_000, 41_000, 35_000, 29_000],
    "market_cap": [255_000, 210_000, 175_000, 138_000, 101_000, 82_000],
    "net_debt": [35_000, 28_000, 22_000, 18_000, 13_000, 11_000],
    "shares_outstanding": [2_650, 1_900, 1_400, 1_150, 900, 760],
    "expected_eps_growth": [0.055, 0.052, 0.047, 0.043, 0.038, 0.035],
})

peers["enterprise_value"] = peers["market_cap"] + peers["net_debt"]
peers["share_price"] = peers["market_cap"] / peers["shares_outstanding"]
peers["eps"] = peers["net_income"] / peers["shares_outstanding"]
peers["pe"] = peers["share_price"] / peers["eps"]
peers["ev_ebitda"] = peers["enterprise_value"] / peers["ebitda"]
peers["ev_ebit"] = peers["enterprise_value"] / peers["ebit"]
peers["price_sales"] = peers["market_cap"] / peers["revenue"]
peers["price_book"] = peers["market_cap"] / peers["book_equity"]
peers["peg"] = peers["pe"] / (peers["expected_eps_growth"] * 100)

peers.round(2)

The target company is excluded when calculating peer medians. Otherwise, we would use the company we are trying to value as part of its own benchmark.

In [ ]:
target = peers.iloc[0]
peer_group = peers.iloc[1:].copy()

multiple_columns = ["pe", "ev_ebitda", "ev_ebit", "price_sales", "price_book", "peg"]
peer_medians = peer_group[multiple_columns].median().to_frame("peer_median")
target_multiples = target[multiple_columns].to_frame("target")
comparison = target_multiples.join(peer_medians)
comparison["target_premium_discount"] = comparison["target"] / comparison["peer_median"] - 1

comparison.round(2)

In [ ]:
def implied_price_from_pe(net_income, shares_outstanding, pe_multiple):
    equity_value = net_income * pe_multiple
    return equity_value / shares_outstanding


def implied_price_from_ev_multiple(metric, net_debt, shares_outstanding, ev_multiple):
    enterprise_value = metric * ev_multiple
    equity_value = enterprise_value - net_debt
    return equity_value / shares_outstanding


def implied_price_from_price_multiple(metric, shares_outstanding, price_multiple):
    equity_value = metric * price_multiple
    return equity_value / shares_outstanding

implied_prices = pd.Series({
    "P/E": implied_price_from_pe(target["net_income"], target["shares_outstanding"], peer_medians.loc["pe", "peer_median"]),
    "EV/EBITDA": implied_price_from_ev_multiple(target["ebitda"], target["net_debt"], target["shares_outstanding"], peer_medians.loc["ev_ebitda", "peer_median"]),
    "EV/EBIT": implied_price_from_ev_multiple(target["ebit"], target["net_debt"], target["shares_outstanding"], peer_medians.loc["ev_ebit", "peer_median"]),
    "P/S": implied_price_from_price_multiple(target["revenue"], target["shares_outstanding"], peer_medians.loc["price_sales", "peer_median"]),
    "P/B": implied_price_from_price_multiple(target["book_equity"], target["shares_outstanding"], peer_medians.loc["price_book", "peer_median"]),
})

valuation_range = implied_prices.describe()[["min", "25%", "50%", "75%", "max"]]

print(f"Current target share price: CHF {target['share_price']:,.2f}")
print(f"Median implied share price: CHF {implied_prices.median():,.2f}")

implied_prices.to_frame("implied_share_price_chf").round(2)

## 4. Visualization

The most useful relative valuation charts show whether the target trades at a premium or discount to comparable companies, and whether the implied valuation range is tight or wide.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_data = peers.set_index("company")[["pe", "ev_ebitda"]]
plot_data.plot(kind="bar", ax=axes[0], color=["#2f6f8f", "#9a6b2f"])
axes[0].set_title("Peer Trading Multiples")
axes[0].set_xlabel("")
axes[0].set_ylabel("Multiple")
axes[0].tick_params(axis="x", rotation=35)
axes[0].legend(["P/E", "EV/EBITDA"])

implied_prices.sort_values().plot(kind="barh", ax=axes[1], color="#2f6f8f")
axes[1].axvline(target["share_price"], color="#9a6b2f", linestyle="--", label="Current price")
axes[1].set_title("Implied Target Share Price by Multiple")
axes[1].set_xlabel("CHF per share")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.scatter(peer_group["expected_eps_growth"], peer_group["pe"], s=90, color="#2f6f8f", label="Peers")
ax.scatter(target["expected_eps_growth"], target["pe"], s=130, color="#9a6b2f", label="Target")

for _, row in peers.iterrows():
    ax.annotate(row["company"], (row["expected_eps_growth"], row["pe"]), xytext=(5, 5), textcoords="offset points", fontsize=8)

ax.set_title("P/E Multiple vs Expected EPS Growth")
ax.set_xlabel("Expected EPS growth")
ax.set_ylabel("P/E multiple")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
ax.legend()

plt.tight_layout()
plt.show()

## 5. Application

Relative valuation is commonly used in equity research, investment banking, and screening. It is fast, market-aware, and easy to communicate.

When applying it to a real company such as Nestle, ask:

- Are the peers truly comparable in product mix, geography, margins, and growth?
- Are the multiples based on historical results or forward estimates?
- Is one company more leveraged than another?
- Are accounting differences affecting EBITDA, EBIT, or net income?
- Does the target deserve a premium because of higher quality, lower risk, or stronger brands?

The best use of relative valuation is not to produce one exact price. It creates a market-based range that can be compared with DCF output.

In [ ]:
current_price = target["share_price"]
median_implied_price = implied_prices.median()
upside_downside = median_implied_price / current_price - 1

print(f"Current target price: CHF {current_price:,.2f}")
print(f"Relative valuation median: CHF {median_implied_price:,.2f}")
print(f"Implied upside/downside: {upside_downside:.1%}")
print(f"Implied range: CHF {implied_prices.min():,.2f} to CHF {implied_prices.max():,.2f}")

## 6. Reflection

- Relative valuation compares market prices across similar companies.
- Multiples are shortcuts, not complete valuation models.
- Enterprise value multiples are better when capital structures differ.
- Equity multiples such as P/E are intuitive but can be distorted by leverage or one-off earnings.
- Peer selection often matters more than the formula.
- A good valuation process compares relative valuation with DCF, historical multiples, and business quality.

Questions to answer after running the notebook:

1. Which multiple gives the highest implied price, and why might that happen?
2. Does the target trade at a premium or discount to peers?
3. Which peer looks least comparable, and what would you remove or adjust?
4. Would you trust a relative valuation more or less than a DCF? Why?